# EduAI Assistant — MCQ Test Generator

---

In [ ]:
!pip install pdfplumber spacy transformers torch sentencepiece groq -q
!python -m spacy download en_core_web_sm -q

import spacy
import re
import json
import random
from collections import Counter

print('All dependencies installed!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 78.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 108.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All dependencies installed!


In [ ]:
import pdfplumber

def extract_text_from_pdf(file_path):
    full_text = ''
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                full_text += text + '\n'
    return full_text

def clean_text(text):
    lines = text.split('\n')
    cleaned = []
    noise = ['whatsapp:', 'megalecture', 'mega lecture', 'email:',
             'www.youtube', 'www.megalecture', 'youtube.com']
    for line in lines:
        s = line.strip()
        if not s: continue
        if any(p in s.lower() for p in noise): continue
        if s.startswith('http') or s.startswith('www.'): continue
        if re.match(r'^\d{1,3}$', s): continue
        cleaned.append(s)
    text = '\n'.join(cleaned)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = re.sub(r'(?i)whatsapp\s*:\s*', '', text)
    text = re.sub(r'(?i)page\s+\d+\s+of\s+\d+', '', text)
    text = re.sub(r'(?i)mega\s*lecture', '', text)
    text = re.sub(r'(?i)refined\s+by\s+\w+', '', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()

from google.colab import files
print('Upload a PDF file:')
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

raw_text = extract_text_from_pdf(file_name)
cleaned_text = clean_text(raw_text)

print(f'\nExtracted and cleaned: {len(cleaned_text)} characters')
print(f'First 300 chars: {cleaned_text[:300]}')

Upload a PDF file:


Saving IGCSE-Biology-Notes.pdf to IGCSE-Biology-Notes.pdf

Extracted and cleaned: 49991 characters
First 300 chars: IGCSE
B i o l o g y
m
o
No t es
c
.
e
r
u
t
c
e
l
a
g
e
m
.
w
w
w

Unit 1 : Characteristics of living
things

Biology is the study of living organisms. For something to be alive it needs to perform all seven
functions of living things.MRS GREN
Movement, Respiration, Sensitivity, Growth, Reproduction


In [ ]:
nlp = spacy.load('en_core_web_sm')
doc = nlp(cleaned_text[:50000])

# Extract and filter key terms
STOP_WORDS = {'the', 'this', 'that', 'which', 'they', 'them', 'what', 'where',
              'when', 'how', 'who', 'why', 'also', 'just', 'very', 'much', 'many', 'some'}

def is_valid(term):
    lower = term.lower().strip()
    if len(lower) < 3: return False
    if any(s in lower for s in STOP_WORDS): return False
    if re.match(r'^[\d\s\.\-\+]+$', lower): return False
    return True

entities = [ent.text.strip() for ent in doc.ents if is_valid(ent.text)]
chunks = [c.text.strip() for c in doc.noun_chunks if len(c.text.strip()) > 3
          and len(c.text.split()) <= 5 and is_valid(c.text)]

term_counts = Counter(entities + chunks)
key_concepts = [(t, c) for t, c in term_counts.most_common(30) if c >= 2]

print('=' * 60)
print('  KEY CONCEPTS IDENTIFIED FOR MCQ GENERATION')
print('=' * 60)
for i, (term, count) in enumerate(key_concepts[:20]):
    print(f'  {i+1:2d}. {term:35s} (frequency: {count})')
print(f'\nTotal key concepts: {len(key_concepts)}')

  KEY CONCEPTS IDENTIFIED FOR MCQ GENERATION
   1. water                               (frequency: 26)
   2. two                                 (frequency: 21)
   3. blood                               (frequency: 17)
   4. Blood                               (frequency: 14)
   5. light                               (frequency: 12)
   6. oxygen                              (frequency: 11)
   7. food                                (frequency: 9)
   8. Unit                                (frequency: 7)
   9. energy                              (frequency: 7)
  10. a number                            (frequency: 7)
  11. xylem                               (frequency: 6)
  12. enzymes                             (frequency: 6)
  13. a reign                             (frequency: 6)
  14. carbon dioxide                      (frequency: 6)
  15. cells                               (frequency: 5)
  16. proteins                            (frequency: 5)
  17. Bacteria                       

In [ ]:
DIFFICULTY_PROMPTS = {
    'recall': (
        'ALL questions must be RECALL level. Test basic factual knowledge. '
        'Use: "What is...?", "Which of the following...?", "Define...", "Name the..."'
    ),
    'comprehension': (
        'ALL questions must be COMPREHENSION level. Test understanding, not memorization. '
        'Use: "Why does...?", "Explain why...", "What is the difference between X and Y?", '
        '"How does X relate to Y?", "Compare X and Y"'
    ),
    'application': (
        'ALL questions must be APPLICATION level. Test ability to apply knowledge. '
        'Use: "A student observes... what is happening?", '
        '"In an experiment where... what would you expect?", '
        '"Given that X happens, which process is responsible?"'
    ),
    'all': (
        'Create a MIX: 3-4 RECALL ("What is...?"), '
        '3-4 COMPREHENSION ("Why does...?", "Compare..."), '
        '2-3 APPLICATION ("A student observes..."). Label each.'
    ),
}

def build_mcq_prompt(text, num_questions, difficulty):
    diff_instruction = DIFFICULTY_PROMPTS.get(difficulty, DIFFICULTY_PROMPTS['all'])
    return f"""Based on the following educational text, generate exactly {num_questions} multiple-choice questions.

{diff_instruction}

Rules:
- Each question must have exactly 4 options
- Only ONE option should be correct
- Wrong options should be plausible but clearly incorrect
- Include a brief explanation for each correct answer

Text:
{text[:4500]}

Return ONLY a valid JSON array:
[
  {{
    "question": "Your question?",
    "options": ["Option A", "Option B", "Option C", "Option D"],
    "correctAnswer": 0,
    "difficulty": "{difficulty}",
    "explanation": "Why this is correct"
  }}
]"""

print('Bloom\'s Taxonomy prompts configured for 3 difficulty levels ')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print('Loading FLAN-T5-base...')
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
print(f'Loaded on {device}')

def generate_local(prompt, max_length=200):
    inputs = tokenizer(prompt, max_length=512, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_length, num_beams=4,
                                  no_repeat_ngram_size=3, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Generate recall questions using FLAN-T5
print('\n' + '=' * 60)
print('  FLAN-T5 MCQ GENERATION (LOCAL MODEL)')
print('=' * 60)

def get_context(text, term, window=400):
    idx = text.lower().find(term.lower())
    if idx == -1: return ''
    return text[max(0,idx-window):min(len(text),idx+len(term)+window)]

flan_questions = []
for term, count in key_concepts[:8]:
    context = get_context(cleaned_text, term)
    if not context: continue

    q = generate_local(f'Generate a factual question about "{term}":\n{context[:500]}\nQuestion:')
    a = generate_local(f'Answer: {q}\nContext: {context[:500]}\nAnswer:')

    if len(q.strip()) > 10 and len(a.strip()) > 3:
        flan_questions.append({'question': q.strip(), 'answer': a.strip(), 'term': term})
        print(f'\n📝 Q: {q.strip()}')
        print(f'   A: {a.strip()}')

print(f'\nFLAN-T5 questions generated: {len(flan_questions)}')

Loading FLAN-T5-base...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded on cuda

  FLAN-T5 MCQ GENERATION (LOCAL MODEL)

📝 Q: How does respiration work?
   A: O2 & glucose breaking

📝 Q: How many organisms are there in a phylum?
   A: Many Organism

📝 Q: How do guard cells interact with each other?
   A: Allows O2 and CO2 to pass

📝 Q: How do guard cells work?
   A: Allows O2 and CO2 to pass

📝 Q: How does light change the state of matter?
   A: respiration

📝 Q: How does respiration work?
   A: IT IS VITAL for survival

📝 Q: How does respiration work?
   A: IT IS VITAL FOR SUFFICIENT

📝 Q: How do living organisms perform their functions?
   A: Movement, Respiration

FLAN-T5 questions generated: 8

Note: FLAN-T5-base (248M params) has limited question quality.
In production, we use Mistral 7B (quantized) for much better local results.


## Fallback method : Generate MCQs with Groq API (Cloud)


In [ ]:
GROQ_API_KEY = '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'

all_groq_questions = {}

if GROQ_API_KEY:
    from groq import Groq
    client = Groq(api_key=GROQ_API_KEY)

    def generate_groq(prompt, system_prompt, max_tokens=2000):
        response = client.chat.completions.create(
            model='llama-3.1-8b-instant',
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt}
            ],
            max_tokens=max_tokens,
            temperature=0.3,
        )
        return response.choices[0].message.content

    def parse_mcqs(response):
        cleaned = response.strip()
        if '```' in cleaned:
            for part in cleaned.split('```'):
                part = part.strip()
                if part.startswith('json'): part = part[4:].strip()
                if part.startswith('['): cleaned = part; break
        start = cleaned.find('[')
        end = cleaned.rfind(']') + 1
        if start != -1 and end > start:
            return json.loads(cleaned[start:end])
        return []

    for difficulty in ['recall', 'comprehension', 'application']:
        print(f'\n{"=" * 60}')
        print(f'  {difficulty.upper()} QUESTIONS (Bloom\'s Taxonomy)')
        print(f'{"=" * 60}')

        prompt = build_mcq_prompt(cleaned_text, 5, difficulty)
        response = generate_groq(
            prompt,
            'You are an expert exam creator. Return ONLY valid JSON.',
        )

        questions = parse_mcqs(response)
        all_groq_questions[difficulty] = questions

        for i, q in enumerate(questions):
            correct_idx = q.get('correctAnswer', 0)
            options = q.get('options', [])
            print(f'\n  Q{i+1}: {q["question"]}')
            for j, opt in enumerate(options):
                marker = '  ✅' if j == correct_idx else ''
                print(f'    {chr(65+j)}) {opt}{marker}')
            if q.get('explanation'):
                print(f'    💡 {q["explanation"]}')

        print(f'\n  Generated: {len(questions)} {difficulty} questions')

else:
    print('Groq API key wrong')

Enter your Groq API key (or press Enter to skip): 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

  RECALL QUESTIONS (Bloom's Taxonomy)

  Q1: What is the study of living organisms called?
    A) Chemistry
    B) Physics
    C) Biology  ✅
    D) Mathematics
    💡 Biology is the study of living organisms.

  Q2: Which of the following is a type of nutrition where organisms make their own food?
    A) Autotrophic nutrition  ✅
    B) Heterotrophic nutrition
    C) Photosynthesis
    D) Respiration
    💡 Autotrophic nutrition refers to organisms that make their own food, such as plants.

  Q3: Name the main groups of living organisms that do not include viruses.
    A) Bacteria, Fungi, Plants, Animals
    B) Bacteria, Pcrotoctista, Fungi, Plants
    C) Bacteria, Fungi, Plants, Animals, Viruses
    D) Bacteria, Pcrotoctista, Fungi, Plants, Animals  ✅
    💡 The five kingdoms of living organisms are Bacteria, Pcrotoctista, Fungi, Plants, and Animals.

  Q4: Define the term 'sensitiv